# Setup

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Walk up until we find the project root (the folder containing data/raw)
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
EXTERNAL = PROJECT_ROOT / "data" / "external"

FILES = {
    "iod_imd": EXTERNAL / "iod2025_imd.csv",
    "iod_domains": EXTERNAL / "iod2025_domains.csv",
    "density": EXTERNAL / "ts006_population_density_lsoa.csv",
    "postcode_lookup": EXTERNAL / "postcode_lookup.csv",
}
for name, path in FILES.items():
    print(f"{name:<16} exists={path.exists()}")

iod_imd          exists=True
iod_domains      exists=True
density          exists=True
postcode_lookup  exists=True


# Look at each area file

In [2]:
data = {}
for name in ["iod_imd", "iod_domains", "density"]:
    df = pd.read_csv(FILES[name], encoding="utf-8-sig")
    data[name] = df

    print("=" * 80)
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("\nColumns:")
    for col in df.columns:
        print("   ", col)
    print("\nFirst 2 rows:")
    print(df.head(2).to_string())
    print()

iod_imd: 33,755 rows x 6 columns

Columns:
    LSOA code (2021)
    LSOA name (2021)
    Local Authority District code (2024)
    Local Authority District name (2024)
    Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)
    Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)

First 2 rows:
  LSOA code (2021)     LSOA name (2021) Local Authority District code (2024) Local Authority District name (2024)  Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)  Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)
0        E01000001  City of London 001A                            E09000001                       City of London                                                                26525                                                                                   8
1        E01000002  City of London 001B                            E09000001                       City of London            

# Join coverage

In [3]:
def find_lsoa_column(df):
    """Return the column where most values look like a 2021 LSOA code (E01 + 6 digits)."""
    for col in df.columns:
        if df[col].astype(str).str.fullmatch(r"E01\d{6}").mean() > 0.9:
            return col
    return None


lookup = pd.read_csv(FILES["postcode_lookup"])
our_lsoas = set(lookup["lsoa21"].dropna())
print(f"Unique LSOAs in our postcode lookup: {len(our_lsoas):,}\n")

for name, df in data.items():
    col = find_lsoa_column(df)
    if col is None:
        print(f"{name}: no LSOA code column found!")
        continue

    codes = set(df[col])
    covered = len(our_lsoas & codes)
    print(f"{name}")
    print(f"   LSOA column:   {col!r}")
    print(f"   Unique codes:  {len(codes):,}")
    print(f"   Coverage:      {covered:,} of {len(our_lsoas):,} ({covered / len(our_lsoas):.1%})")
    print(f"   Not matched:   {sorted(our_lsoas - codes)[:5]}\n")

Unique LSOAs in our postcode lookup: 4,711

iod_imd
   LSOA column:   'LSOA code (2021)'
   Unique codes:  33,755
   Coverage:      4,711 of 4,711 (100.0%)
   Not matched:   []

iod_domains
   LSOA column:   'LSOA code (2021)'
   Unique codes:  33,755
   Coverage:      4,711 of 4,711 (100.0%)
   Not matched:   []

density
   LSOA column:   'geography code'
   Unique codes:  35,672
   Coverage:      4,711 of 4,711 (100.0%)
   Not matched:   []

